In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

## RAG절차

- 1.docx문서를 읽는다(python-docx) : \n은 제외
- 2.읽어온 문서를 쪼갠다.(tiktoken이용)
    * 모델의 context window와 비용확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임배딩 -> vector Store에 저장 => chroma(local파일)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성(LangChain)

# 0. 패키지 설치

In [ ]:
# 문서읽어오기
# pip install -q doc2txt

In [ ]:
# text를 chunk로 나누는 기능(cf.)
%pip install -q langchain-text-splitters

In [ ]:
# 파일저장 chroma(cf. chromadb와다름)
%pip install -q langchain-chroma

In [ ]:
# langchain사용
%pip install -q langchain

# 1. 문서읽기 (x)

In [ ]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tex_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

# 2. 문서를 쪼개면서 읽기(o)
## 2.1 1500토큰씩 겹처서 쪼개면서읽기

In [3]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import TokenTextSplitter
loader = Docx2txtLoader('./tex_docx/소득세법(법률)(제21221호)(20260701).docx')
# gpt-4o-mini 모델의 토큰 추출방식 : cl100k_base 방식
# gpt-4 모델의 토큰 추출방식 : o200k_base방식
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base", # 토큰을 세는 방식 이름
    chunk_size=1500,             # 1500토큰 기준으로 chunk 분리
    chunk_overlap=200,
    # separator 파라미터가 없어
)
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 겹쳐 쪼개면서 읽어오는 시간:',runtime)
print('chunk갯수:',len(documents))

문서를 겹쳐 쪼개면서 읽어오는 시간: 3.5741119384765625
chunk갯수: 180


In [ ]:
# 각 chunk별 글자수
print([len(document.page_content) for document in documents])

In [ ]:
documents[0].page_content

In [ ]:
documents[1].page_content

In [ ]:
documents[2].page_content

In [ ]:
documents[3].page_content

In [ ]:
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(max([len(document.page_content) for document in documents]))
print(min([len(document.page_content) for document in documents]))

## 2.2 1500글자씩 겹쳐 쪼개 읽어오기

In [2]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=200, # 200문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print('문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 :',runtime)
print('chunk갯수:',len(documents))

C:\Users\Admin\AppData\Local\Temp\ipykernel_21876\1643808671.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


문서를 1500글자로 겹쳐 쪼개면서 읽는 시간 : 3.7992029190063477
chunk갯수: 195


In [ ]:
documents[2].page_content

In [ ]:
documents[1].page_content

In [ ]:
# chunk들의 글자수
print([len(document.page_content)for document in documents])

# 3. 쪼갠 문서를 임배딩 ->벡터 데이터 베이스(파일)에 저장
- 임베딩 모델 : upstage의 embedding-query,solar-embedding-1-large-passage
- vector DB 저장 폴더 : ./chroma

In [4]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeding = UpstageEmbeddings(
    #api_key = os.getenv('OPENAI_API_KEY'),
    model = 'solar-embedding-1-large-passage'
)

In [ ]:
# embed_query() : 한 문자욜 임베딩 벡터로 전환
len(embeding.embed_query("소득세법은 어쩌구 저쩌구"))

In [ ]:
# embed_document() : 여러 문자열을 임베딩 백터로 전환
vectors = embeding.embed_documents([
    documents[0].page_content, documents[1].page_content
])


In [ ]:
print(len(vector[0]),len(vector[1]))
print

In [7]:
%%time
from langchain_chroma import Chroma
# 데이터를 처음 저장할때
# database = Chroma.from_documents(
#     documents=documents,# chunking 작업한 documents
#     embedding=embeding,# 임베딩 객체
#     collection_name='tax-collection', # 생략시 이름이 랜덤
#     persist_directory='./chroma_upstage'      # 생략시 로컬에 저장 안 됨. 인메모리 vector DB
# )
# 저장된 vector DB를 가져올 때
database = Chroma(
    embedding_function=embeding,
    collection_name='tax-collection',
    persist_directory='./chroma_upstage'
)

CPU times: total: 15.6 ms
Wall time: 12.8 ms


In [ ]:
database.similarity_search(query="연봉 5천만원인 직장인의 소득세는?")

In [ ]:
# 확인
results = database.get(include=['embeddings','documents','metadatas'])
print('데이터수:',len(results['ids']))
print('문서 임베딩 차원 수 :', len(results['embeddings'][0]))
print('1번째 원본:',results['documents'][1][:100])
print('1번째 원본의 임베딩 벡터 :',results['embeddings'][1][:10])
print('1번째 metabase:',results['meta'])

# 4.vector DB에 질문과 유사도 검색(retrieval)

In [8]:
query = "연봉 5천만원인 직장인의 소득세는 얼마에요?"
retrieved_docs = database.similarity_search(query=query,
                                           k=2)# 기본 k값은 4

In [9]:
len(retrieved_docs)

2

In [11]:
# 검색된 문서를 LLM에 전달하기 위한 가공
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])


# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성 -1

In [ ]:
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model='gpt-4.1-nano')



In [15]:
from langchain_upstage import ChatUpstage
load_dotenv()
llm = ChatUpstage(model='solar-pro4')

In [16]:
prompt = f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참조해서 사용자의 질문에 답변해주세요. context는 다음과 같습니다.
{retrieved_doc}
질문 : {query}
"""
ai_message=llm.invoke(prompt)


In [17]:
ai_message.usage_metadata

{'input_tokens': 1843,
 'output_tokens': 566,
 'total_tokens': 2409,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [18]:
print(ai_message.content)

연봉 5천만 원인 직장인의 소득세를 정확히 계산하려면 근로소득공제, 인적공제, 연금보험료 공제 등 다양한 공제 항목을 적용해야 합니다. 다만, 귀하가 제공해주신 **근로소득세액공제(제59조 제2항)** 기준만을 바탕으로 일반적인 경우를 추정해 보면 다음과 같습니다.

### 1. 근로소득세액공제 금액 (제공된 context 기준)
총급여액이 5천만 원인 경우, 제공된 누진공제식의 두 번째 구간에 해당합니다.
*   **해당 구간**: 총급여액 3,300만 원 초과 ~ 7천만 원 이하
*   **공제 공식**: 74만 원 - [(총급여액 - 3,300만 원) × 8/1000]
*   **계산**:
    *   5,000만 원 - 3,300만 원 = 1,700만 원
    *   1,700만 원 × 0.008 = 13만 6천 원
    *   74만 원 - 13만 6천 원 = **60만 4천 원**
*   **결론**: 제공된 기준상 귀하의 근로소득세액공제액은 **60만 4천 원**이며, 이는 하한액(66만 원)보다 적고 상한액 규정을 따릅니다.

### 2. 실제 소득세 추정 시 고려사항
실제 납부 세액은 근로소득세액공제 외에도 다른 요소들에 의해 결정되므로, 단순히 위 공제액만으로 최종 세금을 단정하기는 어렵습니다.

*   **과세표준 구간**: 연봉 5천만 원인 경우, 근로소득공제를 적용한後の 과세표준은 대략 **1,200만 원 초과 ~ 4,600만 원 이하** 구간에 해당할 가능성이 높습니다. 이 구간의 기본 소득세율은 **15%**(누진공제 126만 원 적용)입니다.
*   **다른 세액공제**: 제공해주신 context에 있는 **자녀세액공제(제59조의2)**나 **연금계좌세액공제(제59조의3)** 등이 적용되면 산출세액에서 추가로 공제됩니다. 예를 들어, 자녀 1명이 있다면 25만 원, 연금저축 납입액이 있다면 그 금액의 12%(또는 15%)가 추가로 공제됩니다.
*   **원천징수 vs 확정신고**: 직장인의 경우 매월 급여에서 간이세액표에 따라

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변생성 -2


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
promptTemplate = ChatPromptTemplate([
    ("system", "당신은 최고의 한국 소득세법 전문가입니다."),
    ("human", f"""다음 문맥을 참고하여 질문에 답변하세요. 답을 모른다면 모른다고 하세요.
    최대 3문장으로 간결하게 답변하세요.
    질문 : {{question}}
    문맥 : {{context}}
    답변 : """)
])
promptTemplate

In [ ]:
prompt = promptTemplate.invoke({'context':retrieved_doc,
                               'question':query})

In [ ]:
ai_message = llm.invoke(prompt) # 과금

In [ ]:
ai_message.content

In [ ]:
from langchain_core.output_parsers import StrOutputParser
outputParsers = StrOutputParser()
outputParsers.invoke(ai_message)

In [ ]:
#위의 작업을 한눈에
outputParsers.invoke(llm.invoke(promptTemplate.invoke(
    {'context':retrieved_doc,'question':query}
)))

# 6. langchain으로 답변 생성

In [ ]:
rag_chain = promptTemplate | llm | outputParsers
rag_chain.invoke({'context':retrieved_doc,'question':query})

Langchain으로 전달

In [21]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage,UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임배딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model="text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')

embedding = UpstageEmbeddings(model='solar-embedding-1-large-passage')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma_upstage',
    collection_name='tax-collection'
)

# 3. Retriever 생성
# (1)search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity',
                                    search_kwarge={'k':3}) # 유사도가 높은 top-3추출
# (2)search_type='mmr':
# retriever = vectorStore.as_retriever(search_type='mmr',
#                                     search_kwarge={'k':3,
#                                                   'fetch_k':10,
#                                                   'lambda_mult':0.5})# 1에 가까울수록 관련성 깊은 계산
# (3)search_type='similarity_score_threshold' :
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold',
#                                     search_kwarge={'k':3,
#                                                   'score_threshold':0.4})

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다.다음 문맥을 참고하여 질문에 답하세요. 
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)

# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])


In [26]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'제공된 문맥만으로는 연봉 5천만 원에 대한 소득세를 정확히 산출할 수 없습니다. 소득세는 근로소득세액공제 외에도 기본공제, 인적공제, 연금·보험료·의료비 등 각종 소득공제와 세액공제를 적용한 종합소득과세표준에 기본세율(과세표준 구간별 6%~45%)을 적용해 계산하기 때문입니다. 정확한 세액을 확인하려면 총급여액 5천만 원에서 근로소득공제를 차감한 후 다른 소득공제와 세액공제를 모두 반영한 과세표준을 산출해야 하며, 여기서는 관련 공제 항목들이 모두 누락되어 있습니다.'